In [3]:
# Cell 1
import torch
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

In [4]:
# Cell 2
class SyntheticRegressionData:
    """합성 데이터를 만들고 PyTorch DataLoader로 제공한다."""

    def __init__(
        self,
        weights,
        bias,
        noise_std=0.01,
        num_train=1000,
        num_val=1000,
        batch_size=32,
    ):
        self.weights = weights
        self.bias = bias
        self.noise_std = noise_std
        self.num_train = num_train
        self.num_val = num_val
        self.batch_size = batch_size

        number_of_examples = num_train + num_val
        number_of_features = weights.numel()

        # 전체 feature를 표준정규분포에서 생성한다.
        self.X = torch.randn(
            number_of_examples,
            number_of_features,
        )

        # 각 sample에 더할 Gaussian noise
        noise = (
            torch.randn(number_of_examples, 1)
            * noise_std
        )

        # y = Xw + b + epsilon
        self.y = (
            self.X @ weights.reshape(-1, 1)
            + bias
            + noise
        )

    def get_tensorloader(self, tensors, train, indices):
        # 훈련 또는 validation에 해당하는 부분만 선택한다.
        selected_tensors = tuple(
            tensor[indices]
            for tensor in tensors
        )

        # feature와 label을 sample 단위로 묶는다.
        dataset = TensorDataset(*selected_tensors)

        # Dataset의 sample을 batch_size개씩 반환한다.
        return DataLoader(
            dataset=dataset,
            batch_size=self.batch_size,
            shuffle=train,
            num_workers=0,
        )

    def get_dataloader(self, train):
        if train:
            indices = slice(0, self.num_train)
        else: # validation
            indices = slice(self.num_train, None)

        return self.get_tensorloader(
            tensors=(self.X, self.y),
            train=train,
            indices=indices,
        )

    def train_dataloader(self):
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        return self.get_dataloader(train=False)

In [5]:
# Cell 3
true_weights = torch.tensor([2.0, -3.4])
true_bias = 4.2

data = SyntheticRegressionData(
    weights=true_weights,
    bias=true_bias,
    noise_std=0.01,
    num_train=1000,
    num_val=1000,
    batch_size=32,
)

train_loader = data.train_dataloader()
val_loader = data.val_dataloader()

In [6]:
# Cell 4
# iter()로 batch iterator를 만들고
# next()로 첫 번째 훈련 batch를 꺼낸다.
train_iterator = iter(train_loader)
batch_X, batch_y = next(train_iterator)

print("Batch X shape:", batch_X.shape)
print("Batch y shape:", batch_y.shape)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))

assert batch_X.shape == (32, 2)
assert batch_y.shape == (32, 1)

# 1000 = 31 * 32 + 8이므로 batch는 총 32개다.
assert len(train_loader) == 32
assert len(val_loader) == 32

Batch X shape: torch.Size([32, 2])
Batch y shape: torch.Size([32, 1])
Number of training batches: 32
Number of validation batches: 32
